<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-04/AI_Modul_4.6_Praktikum_Visualisasi_Data_Matplotlib_Seaborn.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 4.6: Praktikum Visualisasi Data (Matplotlib & Seaborn)
## Desain Visual Ilmiah, Paradigma Berorientasi Objek, dan Palet Aksesibel untuk Agribisnis

Notebook ini menyajikan panduan terapan pembuatan visualisasi data tingkat publikasi menggunakan perpaduan **Matplotlib Object-Oriented API** dan fungsi statistik tingkat tinggi **Seaborn** untuk memantau produktivitas, mikroklimat, dan mutu Tandan Buah Segar (TBS) kelapa sawit.

### Agenda Praktikum:
1. **Inisialisasi Lingkungan & Sintesis Data Multi-Dimensi Kebun**
2. **Penerapan Paradigma Berorientasi Objek Matplotlib (`fig, ax`)**
3. **Pembangunan Kanvas Komposit Tiga Panel Ilmiah (Regplot, Boxplot, Heatmap)**
4. **Eksplorasi Fungsi Tingkat Figur (*Figure-Level*) vs Tingkat Sumbu (*Axes-Level*)**
5. **Penerapan Standar Desain Edward Tufte (*Data-Ink Ratio*) & Palet Ramah Buta Warna**
6. **Visualisasi Grid Spasial Kesehatan Kanopi NDVI Kebun Sawit**

In [ ]:
# 1. Inisialisasi Environment & Import Pustaka
import os
os.environ['MPLBACKEND'] = 'Agg'  # Backend non-interaktif

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Menetapkan tema visual baku ilmiah
sns.set_theme(style='ticks', font_scale=1.0)

print(f"Versi Matplotlib : {plt.matplotlib.__version__}")
print(f"Versi Seaborn    : {sns.__version__}")
print("Lingkungan Visualisasi Ilmiah siap digunakan!")

## 2. Sintesis Dataset Operasional Perkebunan Sawit Multi-Dimensi

In [ ]:
np.random.seed(42)
N = 250

blok_id = [f"BLK-{i+1:03d}" for i in range(N)]
afdeling = np.random.choice(['AFD-A', 'AFD-B', 'AFD-C', 'AFD-D'], size=N)
curah_hujan = np.random.uniform(120.0, 360.0, size=N).round(1)
suhu_kanopi = (38.0 - 0.03 * curah_hujan + np.random.normal(0, 1.2, size=N)).round(1)
pupuk_npk = np.random.uniform(2.0, 4.5, size=N).round(2)
ndvi = np.clip(0.45 + 0.001 * curah_hujan + 0.05 * pupuk_npk + np.random.normal(0, 0.03, size=N), 0.40, 0.90).round(3)

# Kadar FFA (%) berbanding terbalik dengan kedisiplinan rotasi panen
kadar_ffa = np.where(
    afdeling == 'AFD-D',
    np.random.normal(3.8, 0.7, size=N),
    np.random.normal(2.4, 0.5, size=N)
).round(2)
kadar_ffa = np.clip(kadar_ffa, 1.2, 5.5)

# Tonase TBS (Ton/Ha)
tonase = (0.04 * curah_hujan + 2.5 * pupuk_npk + 12.0 * ndvi - 0.4 * kadar_ffa + np.random.normal(0, 1.5, size=N)).round(2)
tonase = np.clip(tonase, 12.0, 35.0)

df_visual = pd.DataFrame({
    'Blok_ID': blok_id,
    'Afdeling': afdeling,
    'Curah_Hujan_mm': curah_hujan,
    'Suhu_C': suhu_kanopi,
    'Dosis_NPK_kg': pupuk_npk,
    'Indeks_NDVI': ndvi,
    'Kadar_FFA_pct': kadar_ffa,
    'Tonase_TBS_Ton_Ha': tonase
})

print("Tampilan 5 Baris Pertama Data Visualisasi:")
print(df_visual.head())

## 3. Pembangunan Kanvas Komposit Tiga Panel Ilmiah

Mengintegrasikan fungsi **Axes-Level** Seaborn ke dalam satu kanvas Matplotlib berorientasi objek berukuran $15 \times 5$ inci:
- **Panel 1:** Regresi linier `sns.regplot` dengan pita interval kepercayaan 95%.
- **Panel 2:** Komparasi mutu FFA antar-afdeling `sns.boxplot` dengan palet ramah buta warna.
- **Panel 3:** Matriks korelasi `sns.heatmap` divergen.

In [ ]:
# Membuat Kanvas 1 Baris x 3 Kolom
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3, figsize=(15, 5), dpi=200)
fig.suptitle('Dashboard Diagnostik Kinerja dan Mutu Perkebunan Kelapa Sawit', fontsize=13, fontweight='bold', y=1.02)

# --- PANEL 1: Curah Hujan vs Tonase TBS (sns.regplot) ---
sns.regplot(
    data=df_visual,
    x='Curah_Hujan_mm',
    y='Tonase_TBS_Ton_Ha',
    ax=ax1,
    color='#1D4ED8',
    scatter_kws={'alpha': 0.5, 's': 30, 'edgecolors': 'none'},
    line_kws={'color': '#DC2626', 'linewidth': 2, 'label': 'Tren Regresi (95% CI)'}
)
ax1.set_title('Respon Curah Hujan terhadap Produksi TBS', fontsize=10, fontweight='bold', pad=10)
ax1.set_xlabel('Curah Hujan Bulanan (mm)', fontsize=9)
ax1.set_ylabel('Produksi TBS (Ton/Ha)', fontsize=9)
ax1.legend(loc='upper left', frameon=True, fontsize=8)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- PANEL 2: Mutu FFA per Afdeling (sns.boxplot) ---
sns.boxplot(
    data=df_visual,
    x='Afdeling',
    y='Kadar_FFA_pct',
    hue='Afdeling',
    legend=False,
    ax=ax2,
    palette='colorblind',
    boxprops=dict(alpha=0.8)
)
# Garis batas kritis mutu ekspor PKS (FFA <= 3.5%)
ax2.axhline(3.5, color='#DC2626', linestyle='--', linewidth=1.5, label='Batas Toleransi FFA (3.5%)')
ax2.set_title('Komparasi Mutu Asam Lemak Bebas (FFA)', fontsize=10, fontweight='bold', pad=10)
ax2.set_xlabel('Divisi Afdeling Kebun', fontsize=9)
ax2.set_ylabel('Kadar FFA (%)', fontsize=9)
ax2.legend(loc='upper left', frameon=True, fontsize=8)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# --- PANEL 3: Matriks Korelasi Spearman (sns.heatmap) ---
kolom_numerik = ['Curah_Hujan_mm', 'Suhu_C', 'Dosis_NPK_kg', 'Indeks_NDVI', 'Kadar_FFA_pct', 'Tonase_TBS_Ton_Ha']
mat_korelasi = df_visual[kolom_numerik].corr(method='spearman')

sns.heatmap(
    mat_korelasi,
    ax=ax3,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1.0,
    vmax=1.0,
    cbar_kws={'label': 'Korelasi (r)'},
    linewidths=0.5,
    annot_kws={'size': 7.5}
)
ax3.set_title('Matriks Korelasi Multivariat Fitur Kebun', fontsize=10, fontweight='bold', pad=10)
ax3.set_xticklabels(['Hujan', 'Suhu', 'NPK', 'NDVI', 'FFA', 'Tonase'], rotation=45, ha='right', fontsize=8)
ax3.set_yticklabels(['Hujan', 'Suhu', 'NPK', 'NDVI', 'FFA', 'Tonase'], rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_6_composite.png', dpi=200, bbox_inches='tight')
print("Plot komposit ilmiah berhasil disimpan ke docs/assets/praktikum_4_6_composite.png")

## 4. Visualisasi Grid Spasial Kesehatan Kanopi NDVI Pertanian Presisi

Memetakan nilai kanopi kebun berdimensi $15 \times 15$ blok menggunakan palet perseptual seragam `viridis` untuk mendeteksi zona defisiensi klorofil.

In [ ]:
# Membuat Grid Spasial 15x15 Blok Kebun
np.random.seed(99)
grid_spasial = np.random.uniform(0.60, 0.88, size=(15, 15))

# Suntikkan klaster defisiensi hara/hama (blok 4:8, kolom 3:7)
grid_spasial[4:8, 3:7] -= 0.28
grid_spasial = np.clip(grid_spasial, 0.35, 0.90)

plt.figure(figsize=(8, 6.5), dpi=200)
ax_spasial = sns.heatmap(
    grid_spasial,
    cmap='viridis',  # Palet perseptual seragam & ramah buta warna
    vmin=0.35,
    vmax=0.90,
    cbar_kws={'label': 'Indeks Vegetasi Kanopi (NDVI)'},
    linewidths=0.5,
    linecolor='#F1F5F9'
)

plt.title('Peta Spasial NDVI Kanopi Kelapa Sawit (15x15 Blok Afdeling)', fontsize=11, fontweight='bold', pad=12)
plt.xlabel('Koordinat Kolom Blok (Barat -> Timur)', fontsize=9, fontweight='bold')
plt.ylabel('Koordinat Baris Blok (Utara -> Selatan)', fontsize=9, fontweight='bold')

# Menandai klaster anomali
rect = plt.Rectangle((3, 4), 4, 4, fill=False, edgecolor='#DC2626', linewidth=2.5, linestyle='--')
ax_spasial.add_patch(rect)
plt.text(5.0, 3.7, 'Klaster Stres Klorofil', color='#DC2626', fontweight='bold', fontsize=8.5, ha='center')

plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_6_spatial.png', dpi=200, bbox_inches='tight')
print("Peta spasial NDVI berhasil disimpan ke docs/assets/praktikum_4_6_spatial.png")
print("\nPraktikum Modul 4.6 Selesai dengan Sukses!")